In [5]:
!pip uninstall -y langchain langchain-core langchain-community langchain-google-genai
!pip install langchain==0.1.20 langchain-community==0.0.38 langchain-core==0.1.52 langchain-google-genai==1.0.3 pageindex rank-bm25

Found existing installation: langchain 1.3.1
Uninstalling langchain-1.3.1:
  Successfully uninstalled langchain-1.3.1
Found existing installation: langchain-core 1.4.0
Uninstalling langchain-core-1.4.0:
  Successfully uninstalled langchain-core-1.4.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 961.0 kB/s eta 0:00:00
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [1]:
import os
from google.colab import userdata
from pageindex import PageIndexClient

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
pi_client = PageIndexClient(api_key=userdata.get('PAGEINDEX'))

In [2]:
import requests
from bs4 import BeautifulSoup

urls = [
    "https://am.gs.com/en-int/advisors/insights/article/2025/dollars-shifting-landscape-from-dominance-to-diversification",
    "https://www.cambridge.org/core/journals/international-organization/article/dollar-diminished-the-unmaking-of-us-financial-hegemony-under-trump/67A0051D4C763B1C76089ED957D9D979",
    "https://www.ft.com/content/9d7d6204-4fc7-4f1d-af05-473c3649efcd",
    "https://www.csis.org/analysis/drone-supply-chain-war-identifying-chokepoints-making-drone",
    "https://www.reuters.com/business/autos-transportation/how-nexperia-chip-crisis-upended-auto-supply-chains-again-2025-11-24/",
    "https://www.cambridge.org/core/journals/international-organization/article/decoupling-dilemma-how-us-sanctions-erode-global-economic-governance/8FA0868798CE6033CBAF995F0E763084",
    "https://www.brookings.edu/articles/the-future-of-economic-statecraft",
    "https://carnegieendowment.org/research/2025/02/the-new-geopolitics-of-energy",
    "https://www.cfr.org/backgrounder/dollar-worlds-reserve-currency",
    "https://www.brookings.edu/articles/what-are-the-differences-between-payment-stablecoins-and-tokenized-bank-deposits/",
    "https://www.gzeromedia.com/gzero-world-clips/the-future-of-the-dollar",
    "https://www.brookings.edu/articles/the-international-monetary-and-financial-system-how-to-fit-it-for-purpose/",
    "https://www.brookings.edu/articles/reforms-for-a-21st-century-global-financial-architecture",
    "https://www.brookings.edu/articles/who-has-to-leave-the-federal-reserve-next-2/"
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("Starting...")

with open("my_corpus.txt", "w", encoding="utf-8") as file:
    for i, url in enumerate(urls, 1):
        try:
            response = requests.get(url, headers=headers, timeout=15)

            if response.status_code != 200:
                print(f"Article failed {i} (Status Code: {response.status_code})")
                continue

            if "application/pdf" in response.headers.get("Content-Type", "").lower():
                file.write(f"\n\n--- ARTICLE {i}: PDF Document ---\n\n")
                file.write(f"Source Link: {url}\nThis file format requires a PDF reader library like pypdf. Storing source reference.\n")
                print(f"Article {i} is a direct PDF link. Saved reference.")
                continue

            soup = BeautifulSoup(response.content, 'html.parser')

            title_element = soup.find('h1')
            title = title_element.text.strip() if title_element else f"Article {i}"

            file.write(f"\n\n--- ARTICLE {i}: {title} ---\n\n")

            paragraphs = soup.find_all('p')
            paragraph_count = 0

            for p in paragraphs:
                text = p.get_text().strip()
                if len(text.split()) > 10:
                    file.write(text + "\n")
                    paragraph_count += 1

            print(f" Saved Article {i}: {title} ({paragraph_count} paragraphs)")

        except Exception as e:
            print(f" Error downloading Article {i}: {e}")

print("\nProcessing complete.")

Starting...
 Saved Article 1: The Dollar's Shifting Landscape: From Dominance to Diversification (72 paragraphs)
 Saved Article 2: Dollar Diminished: The Unmaking of US Financial Hegemony Under Trump (58 paragraphs)
Article failed 3 (Status Code: 403)
 Saved Article 4: The Drone Supply Chain War: Identifying the Chokepoints to Making a Drone (19 paragraphs)
Article failed 5 (Status Code: 401)
 Saved Article 6: The Decoupling Dilemma: How US Sanctions Erode Global Economic Governance (62 paragraphs)
Article failed 7 (Status Code: 404)
 Saved Article 8: Article 8 (0 paragraphs)
 Saved Article 9: The Dollar: The World’s Reserve Currency (38 paragraphs)
 Saved Article 10: What are the differences between payment stablecoins and tokenized bank deposits? (26 paragraphs)
Article failed 11 (Status Code: 404)
 Saved Article 12: The international monetary and financial system: How to fit it for purpose? (22 paragraphs)
 Saved Article 13: Reforms for a 21st century global financial architecture (

In [3]:
!pip install rank-bm25
import time
from typing import Any, List
from pydantic import ConfigDict
from langchain_core.retrievers import BaseRetriever
from langchain_core.documents import Document
from rank_bm25 import BM25Okapi
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
import re

extracted_docs = []
with open("my_corpus.txt", "r", encoding="utf-8") as file:
    content = file.read()
article_chunks = content.split("\n\n--- ARTICLE ")

for chunk in article_chunks:
    if not chunk.strip():
        continue
    header_end_idx = chunk.find("---\n\n")

    if header_end_idx != -1:
        header_line = chunk[:header_end_idx].strip()
        text_content = chunk[header_end_idx + len("---\n\n"):].strip()
        title_match = re.match(r"^\d+:\s*(.*)", header_line)
        title = title_match.group(1).strip() if title_match else "Unknown Title"

        if text_content:
            metadata = {"title": title}
            extracted_docs.append({"text": text_content, "metadata": metadata})
    else:
        print(f"Warning: Skipping malformed chunk (no header end): {chunk[:100]}...")

print(f"Extracted {len(extracted_docs)} nodes from my_corpus.txt.")

lc_documents = [Document(page_content=doc["text"], metadata=doc["metadata"]) for doc in extracted_docs]
bm25_index = BM25Okapi([doc.page_content.lower().split() for doc in lc_documents])

class BM25Retriever(BaseRetriever):
    index: Any
    documents: List[Document]
    k: int = 3
    model_config = ConfigDict(arbitrary_types_allowed=True)
    def _get_relevant_documents(self, query: str) -> List[Document]:
        return self.index.get_top_n(query.lower().split(), self.documents, n=self.k)

custom_retriever = BM25Retriever(index=bm25_index, documents=lc_documents, k=3)

llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)
prompt = ChatPromptTemplate.from_template("Context:\n{context}\n\nQuestion:\n{input}\n\nAnswer (only from context):")
rag_chain = create_retrieval_chain(custom_retriever, create_stuff_documents_chain(llm, prompt))

question = "How do sanctions impact the global economy and supply chains?"

print("\n--- Testing Vectorless RAG ---")
start_time_rag = time.time()
rag_response = rag_chain.invoke({"input": question})
rag_latency = time.time() - start_time_rag
print(f"RAG Latency: {rag_latency:.2f}s")
print(f"Answer: {rag_response['answer'][:200]}...")

print("\n--- Testing Naive Full-Context ---")
full_corpus_text = "\n\n".join([doc["text"] for doc in extracted_docs])
start_time_naive = time.time()
naive_response = llm.invoke(f"Context:\n{full_corpus_text}\n\nQuestion:\n{question}\n\nAnswer (only from context):")
naive_latency = time.time() - start_time_naive
print(f"Naive Latency: {naive_latency:.2f}s")

print("\n--- Final Comparison Results ---")
if rag_latency < naive_latency - 0.5:
    print(f"RAG was faster by {naive_latency - rag_latency:.2f} seconds.")
elif naive_latency < rag_latency - 0.5:
    print(f"Naive was faster by {rag_latency - naive_latency:.2f} seconds.")
else:
    print("Both approaches had similar latency.")

Extracted 9 nodes from my_corpus.txt.

--- Testing Vectorless RAG ---
RAG Latency: 8.64s
Answer: Based on the provided context, sanctions impact the global economy and supply chains in the following ways:

### **Impact on the Global Economy**
* **Erosion and Fragmentation of the Economic Order:**...

--- Testing Naive Full-Context ---
Naive Latency: 13.04s

--- Final Comparison Results ---
RAG was faster by 4.39 seconds.
